 # NYC Taxi Trip Duration Prediction

## 1. Setup and Library Imports

This section imports all necessary libraries for data manipulation, numerical operations, and machine learning.

In [ ]:
# Scientific libraries
import numpy as np
import pandas as pd

pd.options.display.max_columns = None  # Possible to limit


## 2. Data Loading

We load the training and validation datasets from their respective CSV files.

In [ ]:
def load_data(train_path, val_path):
    """Loads training and validation datasets."""
    # Load the training dataset
    train_df = pd.read_csv(train_path)
    # Load the validation dataset
    val_df = pd.read_csv(val_path)
    print("Data loaded successfully.")
    return train_df, val_df

# Define data paths
train_data_path = '../data/split/train.csv'
val_data_path = '../data/split/val.csv'

# Load data using the function
train, val = load_data(train_data_path, val_data_path)

## 3. Initial Data Cleaning

We perform initial cleaning steps, such as dropping unnecessary 'id' columns.

In [ ]:
def drop_id_columns(df_list):
    """Drops the 'id' column from a list of DataFrames if it exists."""
    for i, df in enumerate(df_list):
        if 'id' in df.columns:
            df_list[i] = df.drop(columns=['id'], axis='columns')
            print(f"'id' column dropped from DataFrame {i+1}.")
    return df_list

# Apply the function to train and val DataFrames
train, val = drop_id_columns([train, val])

## 4. Feature Engineering: Date and Time Features

We convert 'pickup_datetime' to datetime objects and extract various time-based features like hour, day of week, month, and create cyclical encodings and rush hour indicators.

In [ ]:
def engineer_datetime_features(df):
    """Converts 'pickup_datetime' to datetime objects and extracts time-based features."""
    df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
    print("Converted 'pickup_datetime' to datetime objects.")

    df['hour'] = df['pickup_datetime'].dt.hour
    df['day_of_week'] = df['pickup_datetime'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    # Cyclical encoding for hour
    hour_rad = 2 * np.pi * df['hour'] / 24
    df['hour_sin'] = np.sin(hour_rad)
    df['hour_cos'] = np.cos(hour_rad)

    df['month'] = df['pickup_datetime'].dt.month

    # Rush hour (5 PM - 10 PM)
    df['rush_hour'] = df['hour'].between(17, 22).astype(int)

    # Busy hour (8 AM - 6 PM)
    df['busy_hour'] = df['hour'].between(8, 18).astype(int)

    # 31st day of the month indicator
    df['day_of_month'] = df['pickup_datetime'].dt.day.astype(int)

    df['week_day_hour'] = df['hour'] * df['day_of_week']

    print("Engineered various datetime features.")
    return df

# Apply the function to both train and val DataFrames
train = engineer_datetime_features(train)
val = engineer_datetime_features(val)

## 5. Feature Engineering: Log Transformation of Trip Duration

We apply a log1p transformation to the `trip_duration` target variable to handle its skewed distribution, which is common for time-based data. This helps improve model performance.

In [ ]:
def log_transform_trip_duration(df, column_name='trip_duration', new_column_name='log_trip_duration'):
    """Applies log1p transformation to the trip duration column."""
    if column_name in df.columns:
        df[new_column_name] = np.log1p(df[column_name])
        print(f"Log1p transformed '{column_name}' to '{new_column_name}'.")
    else:
        print(f"Column '{column_name}' not found in DataFrame.")
    return df

# Apply the function to both train and val DataFrames
train = log_transform_trip_duration(train)
val = log_transform_trip_duration(val)

## 6. Feature Engineering: Location Clusters (K-Means)

To capture geographical patterns, we use K-Means clustering on the pickup and dropoff coordinates. This creates categorical features representing spatial regions.

In [ ]:
from sklearn.cluster import KMeans

def create_location_clusters(train_df, val_df, n_clusters=20, random_state=42):
    """Applies K-Means clustering to pickup and dropoff coordinates to create location clusters."""
    # Stack all pickup and dropoff coordinates from the training data
    coords = np.vstack([
        train_df[['pickup_longitude', 'pickup_latitude']].values,
        train_df[['dropoff_longitude', 'dropoff_latitude']].values
    ])

    # Initialize and fit KMeans on the stacked coordinates
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10) # n_init added for sklearn > 1.2
    kmeans.fit(coords)

    # Predict clusters for pickup and dropoff locations in train and val datasets
    train_df['pickup_cluster'] = kmeans.predict(train_df[['pickup_longitude', 'pickup_latitude']].values)
    train_df['dropoff_cluster'] = kmeans.predict(train_df[['dropoff_longitude', 'dropoff_latitude']].values)

    val_df['pickup_cluster'] = kmeans.predict(val_df[['pickup_longitude', 'pickup_latitude']].values)
    val_df['dropoff_cluster'] = kmeans.predict(val_df[['dropoff_longitude', 'dropoff_latitude']].values)

    print(f"Created {n_clusters} location clusters using K-Means.")
    return train_df, val_df

# Apply the function to create location clusters
train, val = create_location_clusters(train, val)

## 7. Feature Engineering: Distance Metrics

We calculate various distance metrics between pickup and dropoff locations, including Haversine, Manhattan, and Euclidean distances. These distances provide crucial information about the trip's length and complexity.

In [ ]:
def add_haversine_distance(df, R=6371.0):
    """Calculates the Haversine distance between pickup and dropoff coordinates."""
    # Convert latitude and longitude from degrees to radians
    lat1 = np.radians(df["pickup_latitude"])
    lon1 = np.radians(df["pickup_longitude"])
    lat2 = np.radians(df["dropoff_latitude"])
    lon2 = np.radians(df["dropoff_longitude"])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    # Calculate distance
    df["distance_haversine"] = R * c
    print("Added Haversine distance feature.")
    return df

# Apply Haversine distance calculation to train and val DataFrames
train = add_haversine_distance(train)
val = add_haversine_distance(val)

In [ ]:
def add_manhattan_distance(df, R=6371.0):
    """Calculates the Manhattan (city block) distance between pickup and dropoff coordinates."""
    # Convert latitude and longitude from degrees to radians
    lat1 = np.radians(df["pickup_latitude"])
    lon1 = np.radians(df["pickup_longitude"])
    lat2 = np.radians(df["dropoff_latitude"])
    lon2 = np.radians(df["dropoff_longitude"])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Manhattan distance calculation
    lat_distance = R * np.abs(dlat)
    # Approximation for longitude distance, considering Earth's curvature
    lon_distance = R * np.cos((lat1 + lat2) / 2) * np.abs(dlon)

    df["distance_manhattan"] = lat_distance + lon_distance
    print("Added Manhattan distance feature.")
    return df

# Apply Manhattan distance calculation to train and val DataFrames
train = add_manhattan_distance(train)
val = add_manhattan_distance(val)

In [ ]:
def add_euclidean_distance(df):
    """Calculates the Euclidean distance between pickup and dropoff coordinates."""
    df["distance_euclidean"] = np.sqrt(
        (df["dropoff_longitude"] - df["pickup_longitude"])**2 +
        (df["dropoff_latitude"] - df["pickup_latitude"])**2
    )
    print("Added Euclidean distance feature.")
    return df

# Apply Euclidean distance calculation to train and val DataFrames
train = add_euclidean_distance(train)
val = add_euclidean_distance(val)

## 8. Feature Engineering: Bearing Features

Bearing features (sin and cos of the initial bearing) capture the direction of travel, which can be an important predictor for trip duration, especially in areas with consistent traffic flow patterns.

In [ ]:
def add_bearing_features(df):
    """Calculates bearing (direction of travel) between pickup and dropoff points."""
    # Convert coordinates to radians
    lat1 = np.radians(df["pickup_latitude"])
    lon1 = np.radians(df["pickup_longitude"])
    lat2 = np.radians(df["dropoff_latitude"])
    lon2 = np.radians(df["dropoff_longitude"])

    dlon = lon2 - lon1

    # Calculate components for bearing
    y = np.sin(dlon) * np.cos(lat2)
    x = (
        np.cos(lat1) * np.sin(lat2)
        - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    )

    # Calculate bearing in degrees and then radians
    bearing = np.degrees(np.arctan2(y, x))
    bearing_rad = np.radians(bearing)

    # Add sine and cosine of bearing as features
    df["bearing_sin"] = np.sin(bearing_rad)
    df["bearing_cos"] = np.cos(bearing_rad)
    print("Added bearing sine and cosine features.")
    return df

# Apply bearing feature calculation to train and val DataFrames
train = add_bearing_features(train)
val = add_bearing_features(val)

## 9. Feature Engineering: Log Transformation of Distance Features

Similar to trip duration, distance features can also be skewed. Applying a log1p transformation helps normalize their distribution, which can benefit linear models.

In [ ]:
def log_transform_distance_features(df, distance_cols):
    """Applies log1p transformation to specified distance columns."""
    for col in distance_cols:
        if col in df.columns:
            df[f"log_{col}"] = np.log1p(df[col])
            print(f"Log1p transformed '{col}' to 'log_{col}'.")
    return df

# Define the distance columns to transform
distance_cols = [
    "distance_haversine",
    "distance_manhattan",
    "distance_euclidean",
]

# Apply log transformation to distance features in train and val DataFrames
train = log_transform_distance_features(train, distance_cols)
val = log_transform_distance_features(val, distance_cols)

## 10. Outlier Removal for Trip Duration

We remove outliers from the `trip_duration` column in the training set using a 2-sigma rule based on the mean and standard deviation. This helps prevent extreme values from unduly influencing the model training.

In [ ]:
train["hour"] = train["pickup_datetime"].dt.hour
val["hour"] = val["pickup_datetime"].dt.hour

train["day_of_week"] = train["pickup_datetime"].dt.dayofweek
val["day_of_week"] = val["pickup_datetime"].dt.dayofweek

train["is_weekend"] = train["day_of_week"].isin([5, 6]).astype(int)
val["is_weekend"] = val["day_of_week"].isin([5, 6]).astype(int)

# Cyclical encoding
hour_rad = 2 * np.pi * train["hour"] / 24
train["hour_sin"] = np.sin(hour_rad)
train["hour_cos"] = np.cos(hour_rad)

hour_rad = 2 * np.pi * val["hour"] / 24
val["hour_sin"] = np.sin(hour_rad)
val["hour_cos"] = np.cos(hour_rad)

# add month
train["month"] = train["pickup_datetime"].dt.month
val["month"] = val["pickup_datetime"].dt.month

# Rush hour (5 PM - 10 PM)
train["rush_hour"] = train["hour"].between(17, 22).astype(int)
val["rush_hour"] = val["hour"].between(17, 22).astype(int)

# Busy hour (8 AM - 6 PM)
train["busy_hour"] = train["hour"].between(8, 18).astype(int)
val["busy_hour"] = val["hour"].between(8, 18).astype(int)

# 31st day of the month indicator
train["day_of_month"] = (train["pickup_datetime"].dt.day).astype(int)
val["day_of_month"] = (val["pickup_datetime"].dt.day).astype(int)

train['week_day_hour'] = train['hour'] * train['day_of_week']
val['week_day_hour'] = val['hour'] * val['day_of_week']

In [ ]:
def remove_trip_duration_outliers(df, column_name='trip_duration', sigma=2):
    """Removes outliers from the specified column using the sigma rule."""
    if column_name in df.columns:
        mean_val = df[column_name].mean()
        std_val = df[column_name].std()

        # Filter DataFrame based on the 2-sigma rule
        original_rows = len(df)
        df = df[
            (df[column_name] >= mean_val - sigma * std_val) &
            (df[column_name] <= mean_val + sigma * std_val)
        ]
        removed_rows = original_rows - len(df)
        print(f"Removed {removed_rows} outliers from '{column_name}' (2-sigma rule).")
    else:
        print(f"Column '{column_name}' not found for outlier removal.")
    return df

# Apply outlier removal to the training DataFrame
train = remove_trip_duration_outliers(train)

## 11. Feature Definition

We define the lists of categorical and numerical features that will be used for training the model. This clear separation is important for applying different preprocessing steps.

In [ ]:
categorical_features = ['vendor_id', 'passenger_count', 'store_and_fwd_flag', 'pickup_cluster', 'dropoff_cluster', 'month', 'hour', 'day_of_week', 'day_of_month', 'is_weekend', 'rush_hour', 'busy_hour', 'week_day_hour']
numerical_features   = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'hour_sin', 'hour_cos', 'bearing_sin', 'bearing_cos', 'log_distance_haversine', 'log_distance_manhattan', 'log_distance_euclidean']

training_features = categorical_features + numerical_features

## 12. Data Splitting into Features and Target

We separate the training and validation datasets into features (X) and the target variable (y), which is the `log_trip_duration`.

In [ ]:
def split_features_target(df_train, df_val, training_features, target_feature='log_trip_duration'):
    """Splits DataFrames into features (X) and target (y) for training and validation."""
    X_train = df_train[training_features]
    X_val = df_val[training_features]

    y_train = df_train[target_feature]
    y_val = df_val[target_feature]

    print("Data split into X_train, X_val, y_train, y_val.")
    return X_train, X_val, y_train, y_val

# Apply the function to split the data
X_train, X_val, y_train, y_val = split_features_target(train, val, training_features)

## 13. Preprocessing Pipelines

We define a preprocessing pipeline using `ColumnTransformer` to handle numerical and categorical features separately. Numerical features undergo polynomial transformation and standardization, while categorical features are one-hot encoded.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    PolynomialFeatures,
    StandardScaler,
    OneHotEncoder,
)

def create_preprocessing_pipeline(numerical_features, categorical_features):
    """Creates a ColumnTransformer for preprocessing numerical and categorical features."""
    # Pipeline for numerical features: Polynomial features then Scaling
    numeric_pipeline = Pipeline([
        (
            "poly",
            PolynomialFeatures(
                degree=3,
                interaction_only=True,
                include_bias=False,
            ),
        ),
        ("scaler", StandardScaler()),
    ])

    # ColumnTransformer to apply different transformations to different columns
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_pipeline,
                numerical_features,
            ),
            (
                "cat",
                OneHotEncoder(
                    drop="first",  # Drop the first category to avoid multicollinearity
                    handle_unknown="ignore", # Ignore unknown categories during transform
                    sparse_output=False, # Output a dense array
                ),
                categorical_features,
            ),
        ]
    )
    print("Preprocessing pipeline created.")
    return preprocessor

# Create the preprocessor using the defined features
preprocessor = create_preprocessing_pipeline(numerical_features, categorical_features)

## 14. Apply Preprocessing

We apply the defined `ColumnTransformer` to our training and validation feature sets. `fit_transform` is used for the training data, and `transform` is used for the validation data to prevent data leakage.

In [ ]:
def apply_preprocessing(preprocessor, X_train, X_val):
    """Applies the preprocessing pipeline to training and validation data."""
    # Fit and transform training data
    X_train_processed = preprocessor.fit_transform(X_train)
    # Transform validation data using the fitted preprocessor
    X_val_processed = preprocessor.transform(X_val)
    print("Data preprocessing applied to X_train and X_val.")
    return X_train_processed, X_val_processed

# Apply the preprocessing function
X_train_processed, X_val_processed = apply_preprocessing(preprocessor, X_train, X_val)

## 15. Model Training (Ridge Regression)

We train a Ridge Regression model, which is a linear model that uses L2 regularization to prevent overfitting. The `alpha` parameter controls the strength of this regularization.

In [ ]:
from sklearn.linear_model import Ridge

def train_ridge_model(X_train_processed, y_train, alpha=1.0):
    """Trains a Ridge Regression model."""
    model = Ridge(alpha=alpha)
    model.fit(X_train_processed, y_train)
    print(f"Ridge model trained with alpha={alpha}.")
    return model

# Train the Ridge model
model = train_ridge_model(X_train_processed, y_train, alpha=1)

## 16. Make Predictions

Once the model is trained, we use it to make predictions on both the processed training and validation datasets.

In [ ]:
def make_predictions(model, X_train_processed, X_val_processed):
    """Generates predictions using the trained model on processed data."""
    predictions_train = model.predict(X_train_processed)
    predictions_val = model.predict(X_val_processed)
    print("Predictions generated for training and validation sets.")
    return predictions_train, predictions_val

# Generate predictions
predictions_train, predictions_val = make_predictions(model, X_train_processed, X_val_processed)

## 17. Model Evaluation

We evaluate the model's performance using Root Mean Squared Error (RMSE) and R-squared (R²) metrics on both the training and validation sets. RMSE measures the average magnitude of the errors, while R² indicates the proportion of variance in the dependent variable that is predictable from the independent variables.

In [ ]:
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
)

def evaluate_model(y_true, y_pred, set_name="Dataset"):
    """Evaluates model performance using RMSE and R-squared."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{set_name} RMSE: {rmse:.4f}")
    print(f"{set_name} R²  : {r2:.4f}")
    return rmse, r2

print('-------------------------------------------------------------------')
print('-------------------------------------------------------------------')

# Evaluate on training set
train_rmse, train_r2 = evaluate_model(y_train, predictions_train, "Train")

# Evaluate on validation set
val_rmse, val_r2 = evaluate_model(y_val, predictions_val, "Val")

## Kmean=20, poly=3
* Train RMSE: 0.4088
* Train R²  : 0.7207
-------------------------------------------------------------------
-------------------------------------------------------------------
* Val RMSE: 0.4554
* Val R²  : 0.6759

## 18. Conclusion and Summary

This notebook demonstrates a complete machine learning workflow for predicting NYC taxi trip duration, including data loading, feature engineering, preprocessing, model training, and evaluation. The current model configuration (K-Means=20, PolynomialFeatures degree=3, Ridge Regression) yields the following results:

*   **Train RMSE**: 0.4088
*   **Train R²**: 0.7207
*   **Val RMSE**: 0.4554
*   **Val R²**: 0.6759

These results indicate a reasonable fit, with some potential for further optimization to reduce the gap between training and validation performance, possibly by exploring different model architectures, hyperparameter tuning, or additional feature engineering techniques.